In [1]:
%pip install plotly
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Comparación de candidatos MOPSO (Podas)

Las funciones comparan los tres archivos de resultados (`rmse`, `nasa` y `both`) para un dataset. Usan métricas `test_*`, convierten el NASA sumativo a promedio por motor para mostrarlo en escala lineal y mantienen el valor sumativo medido para aplicar los filtros NASA. Los intervalos bootstrap del 95% se muestran en ambas gráficas. Las estrellas marcan candidatos seleccionados. En la gráfica interactiva, el desplegable superior derecho filtra un candidato junto con sus podas; las etiquetas muestran el alpha de poda.

In [4]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from matplotlib.lines import Line2D


OBJECTIVES = ('rmse', 'nasa', 'both')
OBJECTIVE_COLORS = {
    'rmse': '#2878B5',
    'nasa': '#E07A2D',
    'both': '#3A923A',
}


def _required_number(mapping, key, context):
    if key not in mapping:
        raise KeyError(f'Falta el campo {key!r} en {context}.')
    value = float(mapping[key])
    if not np.isfinite(value):
        raise ValueError(f'El campo {key!r} no es finito en {context}.')
    return value


def _required_interval(mapping, key, context):
    if key not in mapping:
        raise KeyError(f'Falta el campo {key!r} en {context}.')
    interval = mapping[key]
    if not isinstance(interval, (list, tuple)) or len(interval) != 2:
        raise ValueError(f'El intervalo {key!r} no es válido en {context}.')
    lower, upper = (float(value) for value in interval)
    if not np.isfinite(lower) or not np.isfinite(upper):
        raise ValueError(f'El intervalo {key!r} no es finito en {context}.')
    return lower, upper


def _compare_candidate_results(
    dataset_name,
    results_dir='mopso_results',
    test_rul_path=None,
    min_parameters=None,
    max_parameters=None,
    min_rmse=None,
    max_rmse=None,
    min_nasa=None,
    max_nasa=None,
    _plotly=False,
):
    """Grafica candidatos; filtros NASA usan el promedio por motor."""
    dataset_name = str(dataset_name).strip().upper()
    supported_datasets = {f'FD00{i}' for i in range(1, 5)}
    if dataset_name not in supported_datasets:
        choices = ', '.join(sorted(supported_datasets))
        raise ValueError(
            f'Dataset no admitido: {dataset_name!r}. Use uno de: {choices}.'
        )

    filter_bounds = {
        'parameter_count': (min_parameters, max_parameters),
        'rmse': (min_rmse, max_rmse),
        'nasa_score': (min_nasa, max_nasa),
    }
    normalized_bounds = {}
    for metric, (minimum, maximum) in filter_bounds.items():
        converted = []
        for label, value in (('mínimo', minimum), ('máximo', maximum)):
            if value is None:
                converted.append(None)
                continue
            number = float(value)
            if not np.isfinite(number):
                raise ValueError(f'El {label} de {metric} debe ser finito.')
            converted.append(number)
        if converted[0] is not None and converted[1] is not None and converted[0] > converted[1]:
            raise ValueError(f'El mínimo de {metric} no puede superar el máximo.')
        normalized_bounds[metric] = tuple(converted)

    results_dir = Path(results_dir)
    if test_rul_path is None:
        rul_filename = f'RUL_{dataset_name}.txt'
        rul_candidates = (
            Path('data') / 'CMAPSSData' / rul_filename,
            Path('data') / 'backup_root_files' / rul_filename,
            results_dir.parent / 'data' / 'CMAPSSData' / rul_filename,
            results_dir.parent / 'data' / 'backup_root_files' / rul_filename,
        )
        test_rul_path = next(
            (path for path in rul_candidates if path.is_file()), None
        )
        if test_rul_path is None:
            searched = ', '.join(str(path) for path in rul_candidates)
            raise FileNotFoundError(
                f'No se encontró {rul_filename}; rutas comprobadas: {searched}. '
                'Pase test_rul_path explícitamente.'
            )
    else:
        test_rul_path = Path(test_rul_path)
        if not test_rul_path.is_file():
            raise FileNotFoundError(f'No se encontró {test_rul_path}.')

    with test_rul_path.open('r', encoding='utf-8') as rul_file:
        test_engine_count = sum(1 for line in rul_file if line.strip())
    if test_engine_count <= 0:
        raise ValueError(f'No se encontraron motores en {test_rul_path}.')

    records = []
    for objective in OBJECTIVES:
        json_path = results_dir / (
            f'{dataset_name}_mopso_final_candidates_v6_{objective}.json'
        )
        if not json_path.is_file():
            raise FileNotFoundError(
                f'No se encontró {json_path}. Pase la carpeta de resultados '
                "con results_dir si no es 'mopso_results'."
            )

        with json_path.open('r', encoding='utf-8') as json_file:
            package = json.load(json_file)
        if package.get('dataset') != dataset_name:
            raise ValueError(
                f'El dataset declarado en {json_path} no coincide con '
                f'{dataset_name}.'
            )
        if package.get('objective') != objective:
            raise ValueError(
                f'El objetivo declarado en {json_path} no coincide con '
                f'{objective}.'
            )

        candidates = package.get('candidates')
        if not isinstance(candidates, list) or not candidates:
            raise ValueError(f'No hay candidatos en {json_path}.')

        for candidate in candidates:
            candidate_id = candidate.get('candidate_id')
            architecture = candidate.get('architecture', {})
            hidden_sizes = architecture.get('hidden_sizes')
            metrics = candidate.get('test_results', {})
            context = f'{json_path.name}, candidato {candidate_id!r}'
            if not candidate_id:
                raise ValueError(f'Falta candidate_id en {context}.')
            if not isinstance(hidden_sizes, list):
                raise ValueError(f'Falta architecture.hidden_sizes en {context}.')
            if not isinstance(metrics, dict):
                raise ValueError(f'test_results no es válido en {context}.')

            parameter_count = int(
                _required_number(candidate, 'parameter_count', context)
            )
            if parameter_count <= 0:
                raise ValueError(f'parameter_count debe ser positivo en {context}.')

            rmse = _required_number(metrics, 'test_rmse', context)
            nasa = _required_number(metrics, 'test_nasa', context)
            rmse_low = _required_number(metrics, 'test_rmse_ci95_lower', context)
            rmse_high = _required_number(metrics, 'test_rmse_ci95_upper', context)
            nasa_low = _required_number(metrics, 'test_nasa_ci95_lower', context)
            nasa_high = _required_number(metrics, 'test_nasa_ci95_upper', context)
            if rmse_low > rmse_high or nasa_low > nasa_high:
                raise ValueError(f'Los límites de confianza están invertidos en {context}.')
            records.append({
                'objective': objective,
                'candidate_id': str(candidate_id),
                'hidden_sizes': tuple(int(size) for size in hidden_sizes),
                'parameter_count': parameter_count,
                'rmse': rmse,
                'rmse_low': rmse_low,
                'rmse_high': rmse_high,
                'nasa_score': nasa,
                'nasa': nasa / test_engine_count,
                'nasa_low': nasa_low / test_engine_count,
                'nasa_high': nasa_high / test_engine_count,
                'selected_winner': bool(candidate.get('selected_winner', False)),
                'model_type': 'Red completa',
                'alpha': None,
            })

        pruned_paths = (
            results_dir / f'{dataset_name}_mopso_pruned_models_{objective}.json',
            results_dir / f'{dataset_name}_mopso_pruned_models_v6_{objective}.json',
        )
        pruned_json_path = next(
            (path for path in pruned_paths if path.is_file()), None
        )
        if pruned_json_path is None:
            searched = ', '.join(str(path) for path in pruned_paths)
            raise FileNotFoundError(
                f'No se encontró un archivo de modelos podados; rutas comprobadas: {searched}.'
            )

        with pruned_json_path.open('r', encoding='utf-8') as json_file:
            pruned_package = json.load(json_file)
        if pruned_package.get('dataset') != dataset_name:
            raise ValueError(
                f'El dataset declarado en {pruned_json_path} no coincide con '
                f'{dataset_name}.'
            )
        if pruned_package.get('objective') != objective:
            raise ValueError(
                f'El objetivo declarado en {pruned_json_path} no coincide con '
                f'{objective}.'
            )

        pruned_models = pruned_package.get('models')
        if not isinstance(pruned_models, list) or not pruned_models:
            raise ValueError(f'No hay modelos podados en {pruned_json_path}.')

        for model in pruned_models:
            candidate_id = model.get('candidate_id')
            architecture = model.get('architecture', {})
            hidden_sizes = architecture.get('hidden_sizes')
            metrics = model.get('test_results', {})
            context = f'{pruned_json_path.name}, candidato {candidate_id!r}'
            if not candidate_id:
                raise ValueError(f'Falta candidate_id en {context}.')
            if not isinstance(hidden_sizes, list):
                raise ValueError(f'Falta architecture.hidden_sizes en {context}.')
            if not isinstance(metrics, dict):
                raise ValueError(f'test_results no es válido en {context}.')

            parameter_count = int(
                _required_number(model, 'parameter_count', context)
            )
            if parameter_count <= 0:
                raise ValueError(f'parameter_count debe ser positivo en {context}.')

            alpha = _required_number(model, 'pruning_alpha', context)
            rmse = _required_number(metrics, 'test_rmse', context)
            nasa = _required_number(metrics, 'test_nasa', context)
            rmse_low, rmse_high = _required_interval(
                metrics, 'test_rmse_ci95', context
            )
            nasa_low, nasa_high = _required_interval(
                metrics, 'test_nasa_ci95', context
            )
            if rmse_low > rmse_high or nasa_low > nasa_high:
                raise ValueError(f'Los límites de confianza están invertidos en {context}.')
            records.append({
                'objective': objective,
                'candidate_id': str(candidate_id),
                'hidden_sizes': tuple(int(size) for size in hidden_sizes),
                'parameter_count': parameter_count,
                'rmse': rmse,
                'rmse_low': rmse_low,
                'rmse_high': rmse_high,
                'nasa_score': nasa,
                'nasa': nasa / test_engine_count,
                'nasa_low': nasa_low / test_engine_count,
                'nasa_high': nasa_high / test_engine_count,
                'selected_winner': bool(model.get('selected_winner', False)),
                'model_type': 'Poda',
                'alpha': alpha,
            })

    records = [
        record for record in records
        if all(
            (minimum is None or record[metric] >= minimum)
            and (maximum is None or record[metric] <= maximum)
            for metric, (minimum, maximum) in normalized_bounds.items()
        )
    ]
    if not records:
        raise ValueError(
            'Ningún candidato cumple los límites solicitados; amplíe o quite filtros.'
        )

    parameter_logs = np.log10([record['parameter_count'] for record in records])
    log_min = float(parameter_logs.min())
    log_range = float(parameter_logs.max() - log_min)
    bubble_areas = [
        70.0 + 230.0 * ((value - log_min) / log_range if log_range else 0.5)
        for value in parameter_logs
    ]
    rmse_values = np.array([record['rmse'] for record in records])
    nasa_values = np.array([record['nasa'] for record in records])
    x_span = max(float(np.ptp(rmse_values)), float(np.max(np.abs(rmse_values))) * 0.05, 1.0)
    y_span = max(float(np.ptp(nasa_values)), float(np.max(np.abs(nasa_values))) * 0.05, 1.0)
    cap_x = x_span * 0.006
    cap_y = y_span * 0.006

    if _plotly:
        return _plot_candidate_results(
            records, dataset_name, bubble_areas
        )

    fig, ax = plt.subplots(figsize=(13, 8))
    annotation_offsets = ((6, 6), (6, -13), (-6, 8), (-6, -13))
    for index, (record, bubble_area) in enumerate(zip(records, bubble_areas)):
        color = OBJECTIVE_COLORS[record['objective']]
        x, y = record['rmse'], record['nasa']
        ax.hlines(
            y, record['rmse_low'], record['rmse_high'],
            color=color, alpha=0.65, linewidth=1.2, zorder=1,
        )
        ax.vlines(
            [record['rmse_low'], record['rmse_high']], y - cap_y, y + cap_y,
            color=color, alpha=0.65, linewidth=1.2, zorder=1,
        )
        ax.vlines(
            x, record['nasa_low'], record['nasa_high'],
            color=color, alpha=0.65, linewidth=1.2, zorder=1,
        )
        ax.hlines(
            [record['nasa_low'], record['nasa_high']], x - cap_x, x + cap_x,
            color=color, alpha=0.65, linewidth=1.2, zorder=1,
        )
        ax.scatter(
            x, y, s=bubble_area, color=color, alpha=0.72,
            edgecolors='white', linewidths=0.9, zorder=2,
        )
        if record['selected_winner']:
            ax.scatter(
                x, y, s=55, marker='*', color=color,
                edgecolors='black', linewidths=0.6, zorder=4,
            )

        offset = annotation_offsets[index % len(annotation_offsets)]
        ax.annotate(
            (
                f"{record['objective']} {record['hidden_sizes']} · α={record['alpha']:g}"
                if record['alpha'] is not None
                else f"{record['objective']} {record['hidden_sizes']} · sin poda"
            ),
            (x, y), xytext=offset, textcoords='offset points',
            ha='left' if offset[0] > 0 else 'right',
            va='bottom' if offset[1] > 0 else 'top', fontsize=7,
            bbox={'boxstyle': 'round,pad=0.15', 'fc': 'white', 'ec': 'none', 'alpha': 0.72},
            zorder=5,
        )

    method_handles = [
        Line2D(
            [], [], marker='o', linestyle='none', markersize=8,
            markerfacecolor=OBJECTIVE_COLORS[objective], markeredgecolor='white',
            label=objective,
        )
        for objective in OBJECTIVES
    ]
    objective_legend = ax.legend(
        handles=method_handles, title='Optimización', loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
    )
    ax.add_artist(objective_legend)

    unique_counts = sorted({record['parameter_count'] for record in records})
    if len(unique_counts) > 3:
        indices = np.unique(np.round(np.linspace(0, len(unique_counts) - 1, 3)).astype(int))
        size_counts = [unique_counts[index] for index in indices]
    else:
        size_counts = unique_counts
    size_handles = []
    for count in size_counts:
        log_count = np.log10(count)
        fraction = (log_count - log_min) / log_range if log_range else 0.5
        area = 70.0 + 230.0 * fraction
        size_handles.append(
            Line2D(
                [], [], marker='o', linestyle='none', markersize=np.sqrt(area),
                markerfacecolor='none', markeredgecolor='0.4',
                label=f'{count:,}',
            )
        )
    size_legend = ax.legend(
        handles=size_handles, title='Parámetros', loc='upper left',
        bbox_to_anchor=(1.02, 0.68), labelspacing=1.2,
    )
    ax.add_artist(size_legend)
    winner_handle = Line2D(
        [], [], marker='*', linestyle='none', markersize=8,
        markerfacecolor='gold', markeredgecolor='black', label='Seleccionado',
    )
    ax.legend(
        handles=[winner_handle], loc='upper left', bbox_to_anchor=(1.02, 0.27),
        frameon=False,
    )

    ax.set_title(
        f'Comparación de candidatos — {dataset_name}'
    )
    ax.set_xlabel('RMSE de test')
    ax.set_yscale('linear')
    ax.set_ylabel('NASA score medio por motor')
    ax.grid(True, linestyle='--', alpha=0.3)
    fig.tight_layout(rect=(0, 0, 0.78, 1))
    plt.show()
    return fig, ax


def _plot_candidate_results(records, dataset_name, bubble_areas):
    bubble_sizes = [np.sqrt(area) for area in bubble_areas]
    hidden_layer_counts = sorted({len(record['hidden_sizes']) for record in records})
    candidate_ids = sorted({record['candidate_id'] for record in records})
    fig = go.Figure()
    trace_candidate_ids = []

    for hidden_layer_count in hidden_layer_counts:
        layer_legend_label = (
            f'{hidden_layer_count} capa oculta'
            if hidden_layer_count == 1
            else f'{hidden_layer_count} capas ocultas'
        )
        layer_legend_group = f'hidden-layers-{hidden_layer_count}'
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker={'size': 10, 'color': '#777777'},
            name=layer_legend_label, legendgroup=layer_legend_group,
            showlegend=True,
        ))
        trace_candidate_ids.append(None)

        for objective in OBJECTIVES:
            for candidate_id in candidate_ids:
                candidate_data = [
                    record for record in records
                    if record['objective'] == objective
                    and len(record['hidden_sizes']) == hidden_layer_count
                    and record['candidate_id'] == candidate_id
                ]
                if not candidate_data:
                    continue
                candidate_sizes = [
                    size for record, size in zip(records, bubble_sizes)
                    if record['objective'] == objective
                    and len(record['hidden_sizes']) == hidden_layer_count
                    and record['candidate_id'] == candidate_id
                ]
                customdata = [
                    [
                        record['candidate_id'],
                        str(record['hidden_sizes']),
                        record['parameter_count'],
                        record['rmse_low'],
                        record['rmse_high'],
                        record['nasa_low'],
                        record['nasa_high'],
                        record['nasa_score'],
                        objective,
                        record['selected_winner'],
                        (
                            f"Poda (α={record['alpha']:g})"
                            if record['alpha'] is not None else 'Red completa'
                        ),
                    ]
                    for record in candidate_data
                ]
                fig.add_trace(go.Scatter(
                    x=[record['rmse'] for record in candidate_data],
                    y=[record['nasa'] for record in candidate_data],
                    mode='markers+text',
                    text=[
                        (
                            f"{objective} {record['hidden_sizes']} · α={record['alpha']:g}"
                            if record['alpha'] is not None
                            else f"{objective} {record['hidden_sizes']} · sin poda"
                        )
                        for record in candidate_data
                    ],
                    textposition='top right',
                    textfont={'size': 9},
                    customdata=customdata,
                    name=layer_legend_label,
                    legendgroup=layer_legend_group,
                    showlegend=False,
                    marker={
                        'size': [
                            13 if record['selected_winner'] else size
                            for record, size in zip(candidate_data, candidate_sizes)
                        ],
                        'symbol': [
                            'star' if record['selected_winner'] else 'circle'
                            for record in candidate_data
                        ],
                        'color': OBJECTIVE_COLORS[objective],
                        'opacity': 0.72,
                        'line': {'color': 'white', 'width': 1},
                    },
                    error_x={
                        'type': 'data',
                        'symmetric': False,
                        'array': [record['rmse_high'] - record['rmse'] for record in candidate_data],
                        'arrayminus': [record['rmse'] - record['rmse_low'] for record in candidate_data],
                        'color': OBJECTIVE_COLORS[objective],
                        'thickness': 1,
                        'width': 4,
                    },
                    error_y={
                        'type': 'data',
                        'symmetric': False,
                        'array': [record['nasa_high'] - record['nasa'] for record in candidate_data],
                        'arrayminus': [record['nasa'] - record['nasa_low'] for record in candidate_data],
                        'color': OBJECTIVE_COLORS[objective],
                        'thickness': 1,
                        'width': 4,
                    },
                    hovertemplate=(
                        '<b>%{customdata[8]}</b> · candidato %{customdata[0]}'
                        '<br>Arquitectura: %{customdata[1]}'
                        '<br>Parámetros: %{customdata[2]:,.0f}'
                        '<br>RMSE: %{x:.4f} (IC 95%: %{customdata[3]:.4f}–%{customdata[4]:.4f})'
                        '<br>NASA medio por motor: %{y:.4g} '
                        '(IC 95%: %{customdata[5]:.4g}–%{customdata[6]:.4g})'
                        '<br>NASA sumativo: %{customdata[7]:.4g}'
                        '<br>Modelo: %{customdata[10]}'
                        '<extra></extra>'
                    ),
                ))
                trace_candidate_ids.append(candidate_id)

    candidate_buttons = [{
        'label': 'Todos',
        'method': 'update',
        'args': [{'visible': [True] * len(trace_candidate_ids)}],
    }]
    for candidate_id in candidate_ids:
        candidate_buttons.append({
            'label': candidate_id,
            'method': 'update',
            'args': [{
                'visible': [
                    trace_id is None or trace_id == candidate_id
                    for trace_id in trace_candidate_ids
                ]
            }],
        })

    fig.update_layout(
        title=(
            f'Comparación interactiva de candidatos — {dataset_name}'
            '<br><sup>Color: rmse (azul), nasa (naranja), both (verde) · ★ = seleccionado</sup>'
        ),
        xaxis_title='RMSE de test',
        yaxis_title='NASA score medio por motor',
        yaxis_type='linear',
        template='plotly_white',
        width=1200,
        height=760,
        margin={'l': 80, 'r': 300, 't': 150, 'b': 70},
        updatemenus=[{
            'buttons': candidate_buttons,
            'direction': 'down',
            'showactive': True,
            'x': 1.01,
            'y': 1.25,
            'xanchor': 'left',
            'yanchor': 'top',
        }],
        annotations=[{
            'text': 'Filtrar candidato:',
            'x': 1.01,
            'y': 1.29,
            'xref': 'paper',
            'yref': 'paper',
            'showarrow': False,
            'xanchor': 'left',
            'yanchor': 'bottom',
        }],
        legend={
            'title': 'Capas ocultas (clic para filtrar)',
            'x': 1.01,
            'y': 1.02,
            'xanchor': 'left',
            'groupclick': 'togglegroup',
        },
        hoverlabel={'namelength': -1},
    )
    display(HTML(fig.to_html(include_plotlyjs='cdn')))
    return fig


def compare_candidate_results(
    dataset_name,
    results_dir='mopso_results',
    test_rul_path=None,
    min_parameters=None,
    max_parameters=None,
    min_rmse=None,
    max_rmse=None,
    min_nasa=None,
    max_nasa=None,
):
    """Grafica los candidatos de forma interactiva con Plotly."""
    _compare_candidate_results(
        dataset_name, results_dir, test_rul_path,
        min_parameters, max_parameters, min_rmse, max_rmse,
        min_nasa, max_nasa, _plotly=True,
    )

In [5]:
compare_candidate_results('FD001', max_rmse=None, max_nasa=None)

compare_candidate_results('FD003', max_rmse=None, max_nasa=None)